# 🎙️ Sutta TTS Training Control Panel (v14.1)
**Author:** SuttaPlayer | **Mode:** Multi-Account Portable

### 🚀 Quick Start (Account Hop)
1. **Setup:** Run Cell 1 (Mount & Clone).
2. **Download:** Run Cell 2 (Fetch Assets from Drive).
3. **Verify:** Run Cell 3 (Pre-Flight Check).
4. **Resume:** Copy command from Cell 4 to Terminal.
5. **Monitor:** Copy command from Cell 5 to Terminal.
6. **Keep-Alive:** Run Cell 6 (Sheets Sync).

### 🛠️ Architecture
- **Repo:** `sutta-tts-model-training` (Hosts manager, notebook, configs).
- **Data:** `piper_cache.tgz` (Pre-computed features, no WAV transfer).
- **State:** `trainer_state.json` (Tracks epoch/optimizer state).
- **Config:** Updated to use `VOICE_NAME.json` in `piper_training`.

In [ ]:
# ============================================================
# CELL 1: BOOTSTRAP & MOUNT
# ============================================================
from google.colab import drive
import os
import subprocess

# 1. Mount Drive
print("🔗 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# 2. Define Paths
DRIVE_BASE = "/content/drive/MyDrive"
REPO_DIR = f"{DRIVE_BASE}/sutta-tts-model-training"
PIPER_TRAINING = f"{DRIVE_BASE}/piper_training"
LOCAL_CACHE = "/content/piper_cache"
PIPER_REPO = "/content/piper1-gpl"

# 3. Clone/Update Repo (Manages scripts & configs)
print(f"📦 Syncing Repo to {REPO_DIR}...")
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhamma-initiative/sutta-tts-model-training.git {REPO_DIR}
    print("✅ Repo cloned.")
else:
    !cd {REPO_DIR} && git pull
    print("✅ Repo updated.")

# 4. Create Directories
os.makedirs(f"{PIPER_TRAINING}/checkpoints", exist_ok=True)
os.makedirs(LOCAL_CACHE, exist_ok=True)
print("✅ Directories ready.")

# 5. Check Deno
try:
    !deno --version
    print("✅ Deno is installed.")
except:
    print("⚠️  Deno missing. Installing...")
    !curl -fsSL https://deno.land/install.sh | sh
    os.environ['PATH'] += ':/root/.deno/bin'
    !deno --version
    print("✅ Deno installed.")

print("\n🚀 Bootstrap Complete. Proceed to Cell 2 (Download Assets).")

In [ ]:
# ============================================================
# CELL 2: DOWNLOAD REQUIRED FILES
# ============================================================
# These files must exist on Drive for the account hop to work.
# If they are missing, run these commands to fetch them.

print("📥 Downloading Required Assets to Drive...")
print("(Skipping if already present)")

# 1. Pip Environment (Exact Mode)
if not os.path.exists(f"{DRIVE_BASE}/piper_env_pip_list.json"):
    print("Fetching piper_env_pip_list.json...")
    !wget -O "{DRIVE_BASE}/piper_env_pip_list.json" "https://drive.usercontent.google.com/download?id=1poYRO7NjzrzlEpWFuuRRPGIPY5WJdjYM&export=download&confirm=yes"
else:
    print("✅ piper_env_pip_list.json already present.")

# 2. Piper Cache (Pre-computed features)
if not os.path.exists(f"{DRIVE_BASE}/piper_cache.tgz"):
    print("Fetching piper_cache.tgz...")
    !wget -O "{DRIVE_BASE}/piper_cache.tgz" "https://drive.usercontent.google.com/download?id=1SKvjCqIp9vYDWvZebEqpreqGLz4UvmJ4&export=download&confirm=yes"
else:
    print("✅ piper_cache.tgz already present.")

# 3. Last Checkpoint
if not os.path.exists(f"{PIPER_TRAINING}/checkpoints/last.ckpt"):
    print("Fetching last.ckpt...")
    !wget -O "{PIPER_TRAINING}/checkpoints/last.ckpt" "https://drive.usercontent.google.com/download?id=1inHU_oii-A1fp99VvLgBJt4vndRKabl4&export=download&confirm=yes"
else:
    print("✅ last.ckpt already present.")

# 4. Requirements Fallback
if not os.path.exists(f"{DRIVE_BASE}/piper_env_requirements.txt"):
    print("Fetching piper_env_requirements.txt...")
    !wget -O "{DRIVE_BASE}/piper_env_requirements.txt" "https://drive.usercontent.google.com/download?id=1fitiaVPssheC7usoKIFbIaLa579QAP89&export=download&confirm=yes"
else:
    print("✅ piper_env_requirements.txt already present.")

print("\n✅ Asset Download Check Complete. Proceed to Cell 3 (Pre-Flight).")

In [ ]:
# ============================================================
# CELL 3: PRE-FLIGHT CHECK
# ============================================================
# Verifies environment, extensions, and paths before training.

print("🔍 Running Pre-Flight Diagnostics...")

# 1. Run Manager Diagnostics
!deno run --allow-all {REPO_DIR}/scripts/sutta-training-manager.ts --diag-setup

# 2. Verify Critical Files Exist
checks = {
    "Pip Env JSON": f"{DRIVE_BASE}/piper_env_pip_list.json",
    "Piper Cache": f"{DRIVE_BASE}/piper_cache.tgz",
    "Last Checkpoint": f"{PIPER_TRAINING}/checkpoints/last.ckpt",
    "Voice Config": f"{PIPER_TRAINING}/en_gb-suttaplayer-medium.json"
}

all_ok = True
for name, path in checks.items():
    if os.path.exists(path):
        print(f"✅ {name}: OK")
    else:
        print(f"❌ {name}: MISSING at {path}")
        all_ok = False

# 3. Verify MonotonicAlign (Critical for Training)
print("\n🔍 Verifying MonotonicAlign Extension...")
try:
    # We need to ensure PYTHONPATH is set for the check
    import sys
    if PIPER_REPO not in sys.path:
        sys.path.append(PIPER_REPO)
    from piper.train.vits.dataset import MonotonicAlign
    print("✅ MonotonicAlign loaded successfully.")
except ImportError:
    print("⚠️  MonotonicAlign missing! Rebuilding...")
    !cd {PIPER_REPO} && ./build_monotonic_align.sh
    !cd {PIPER_REPO} && python3 setup.py build_ext --inplace
    # Re-verify
    try:
        from piper.train.vits.dataset import MonotonicAlign
        print("✅ MonotonicAlign rebuilt and loaded.")
    except:
        print("❌ CRITICAL: MonotonicAlign failed to build. Training will fail.")
        all_ok = False

if all_ok:
    print("\n✅ ALL CHECKS PASSED. Ready to Train.")
else:
    print("\n❌ CHECKS FAILED. Fix missing files above before proceeding.")

In [ ]:
# ============================================================
# CELL 4: RESUME TRAINING (Command Generator)
# ============================================================
# Copy the command below to the Colab Terminal to start training.

CKPT_PATH = f"{PIPER_TRAINING}/checkpoints/last.ckpt"
BATCH_SIZE = 8  # GPU: 8, CPU: 2
IS_CPU = False  # Set True for CPU testing
MAX_EPOCHS = None # Set to integer to force stop (e.g., 9500)

print("🚀 Generating Training Command...")
print(f"   Checkpoint: {CKPT_PATH}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Mode: {'CPU' if IS_CPU else 'GPU'}")

cmd = [
    f"deno run --allow-all {REPO_DIR}/scripts/sutta-training-manager.ts",
    "--train",
    f"--ckpt-path {CKPT_PATH}",
    f"--batch-size {BATCH_SIZE}"
]

if IS_CPU:
    cmd.append("--cpu")
if MAX_EPOCHS:
    cmd.append(f"--max-epochs {MAX_EPOCHS}")

final_cmd = " ".join(cmd)

print("\n" + "="*60)
print("COPY THIS COMMAND TO THE TERMINAL:")
print("="*60)
print(final_cmd)
print("="*60)

# Also print to console for easy copy
print("\nCommand:", final_cmd)

In [ ]:
# ============================================================
# CELL 5: MONITOR & PRUNE (Command Generator)
# ============================================================
# Copy this command to the terminal to sync checkpoints and prune old ones.

print("🔄 Generating Monitor/Prune Command...")
monitor_cmd = f"deno run --allow-all {REPO_DIR}/sutta-training-manager.ts --monitor"

print("\n" + "="*60)
print("COPY THIS COMMAND TO THE TERMINAL:")
print("="*60)
print(monitor_cmd)
print("="*60)

print("\nNote: Run this periodically to manage Drive quota.")

In [ ]:
# ============================================================
# CELL 6: KEEP-ALIVE (Sheets Sync)
# ============================================================
# Run this cell to keep the Colab session alive and sync metrics.

import os
import time
import pandas as pd
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default

# Paths
METRICS_CSV = f"{PIPER_TRAINING}/uat_metrics.csv"
sheet_name = "SuttaPlayer_UAT_Convergence"

try:
    creds, _ = default()
    gc = gspread.authorize(creds)
    try:
        sh = gc.open(sheet_name)
    except:
        sh = gc.create(sheet_name)
        print(f"✅ Created Sheet: {sheet_name}")

    ws = sh.get_worksheet(0)
    last_row = len(ws.col_values(1))
    
    print(f"💾 Keep-Alive Active. Monitoring CSV...")
    while True:
        # Monitor CSV
        if os.path.exists(METRICS_CSV):
            try:
                df = pd.read_csv(METRICS_CSV)
                if len(df) > last_row:
                    new_data = df.iloc[last_row:].values.tolist()
                    for row in new_data:
                        ws.append_row(row)
                        print(f"  [Sheet] Logged Epoch {row[1]}")
                    last_row = len(df)
            except:
                pass # Ignore write conflicts
        
        time.sleep(60)
except KeyboardInterrupt:
    print("\n⏹️  Sync Stopped.")

In [ ]:
# ============================================================
# CELL 7: BACKUP STATE (For Next Account Hop)
# ============================================================
# Lightweight backup: Only saves trainer_state.json and checkpoint info.
# Cache re-zipping is skipped as dataset is static.

import os
import time
import torch

print("📦 Creating Lightweight Backup for Next Account...")

# 1. Define Paths
DRIVE_BASE = "/content/drive/MyDrive"
PIPER_TRAINING = f"{DRIVE_BASE}/piper_training"
LOCAL_LOGS = "/content/piper1-gpl/lightning_logs"

# 2. Backup Trainer State
state_dst = f"{PIPER_TRAINING}/trainer_state.json"
state_found = False

if os.path.exists(LOCAL_LOGS):
    versions = [d for d in os.listdir(LOCAL_LOGS) if d.startswith("version_")]
    if versions:
        # Get the latest version (sorted alphabetically works for version_0, version_1, etc.)
        latest = sorted(versions)[-1]
        state_file = f"{LOCAL_LOGS}/{latest}/trainer_state.json"
        
        if os.path.exists(state_file):
            !cp {state_file} {state_dst}
            print(f"✅ Trainer State backed up: {state_dst}")
            state_found = True
        else:
            print(f"⚠️ No trainer_state.json found in {LOCAL_LOGS}/{latest}/")
    else:
        print("⚠️ No training versions found in logs.")
else:
    print("⚠️ Local logs directory not found. Training may not have started yet.")

# 3. Extract Checkpoint Info
ckpt_path = f"{PIPER_TRAINING}/checkpoints/last.ckpt"
if os.path.exists(ckpt_path):
    try:
        # Load only metadata (map_location='cpu' is safe)
        checkpoint = torch.load(ckpt_path, map_location='cpu')
        
        if isinstance(checkpoint, dict):
            epoch = checkpoint.get('epoch', 'Unknown')
            global_step = checkpoint.get('global_step', 'Unknown')
            
            info_path = f"{PIPER_TRAINING}/last_checkpoint_info.txt"
            with open(info_path, "w") as f:
                f.write(f"Epoch: {epoch}\n")
                f.write(f"Global Step: {global_step}\n")
                f.write(f"Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            print(f"✅ Checkpoint Info saved: Epoch {epoch}, Step {global_step}")
            print(f"   File: {info_path}")
        else:
            print("⚠️ Checkpoint format unexpected (not a dict).")
    except Exception as e:
        print(f"❌ Error reading checkpoint: {e}")
else:
    print("⚠️ Checkpoint file not found at expected path.")

print("\n✅ Backup Complete. Ready for Account Hop.")